In [2]:
# OpenAI API를 LangChain에서 쉽게 쓸 수 있도록 해주는 래퍼 모듈
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
import json
load_dotenv()

model_type = "gpt-4.1-mini"
CLASSIFIER_PROMPT = """
당신은 사용자의 메시지를 분류하는 분류기입니다.

[분류 기준]
- new_trip: 아래 중 하나라도 해당되면 new_trip
    - 새로운 여행지가 언급됨 (예: 도쿄, 파리, 제주도)
    - 새로운 여행 기간이 언급됨 (예: 3박 4일, 2박 3일)
    - "다시", "새로", "다른 곳", "대신" 같은 표현 포함
    - 여행 계획/일정을 새로 요청하는 문장

- followup: 아래 중 하나라도 해당되면 followup
    - 현재 진행 중인 여행 일정을 수정/보완 요청
    - 현재 여행지에 대한 추가 질문 (날씨, 환율, 맛집 등)
    - "바꿔줘", "수정해줘", "추가해줘", "어때?" 같은 표현

[응답 규칙]
반드시 아래 JSON 형식으로만 응답하세요. 다른 텍스트는 절대 포함하지 마세요.
마크다운 코드 블록 없이 순수 JSON만 응답하세요.
{"intent": "new_trip"} 또는 {"intent": "followup"}
"""
# LCEL 방식 : 파이프라인 형태로 선언
llm = ChatOpenAI(model="gpt-4.1-mini")  # 사용할 OpenAI 모델 지정

agent = create_agent(
        model=llm,
        system_prompt=CLASSIFIER_PROMPT,
        checkpointer=checkpointer if model_type == "gpt-4.1-nano" else None
    )

prompt = "경복궁도 가는 일정으로 수정해줘"

result = agent.invoke(
        {
            "messages" : [
                {"role": "user", "content": prompt}
            ]
        }
    )
test = json.loads(result["messages"][-1].content)
print(test["intent"])

followup


In [21]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver


MAIN_PROMPT = """
당신은 친절하고 전문적인 여행 플래너입니다.
사용자의 요청 앞에 [요약 요청] 또는 [세부 요청] 태그가 붙습니다.
태그에 따라 아래 지침대로 응답하세요.

================================================================
[요약 요청] 태그가 붙은 경우
================================================================
여행 테마를 JSON 형식으로 제안하세요.

- 사용자가 여행 컨셉을 지정하지 않은 경우: 3가지 테마 제안
- 사용자가 여행 컨셉을 지정한 경우: 1~2가지 테마 제안
- 각 테마는 제목과 요약만 포함 (장소 목록은 불필요)
- summary는 markdown 형식으로 작성
    - 핵심 키워드는 **굵게** 표시
    - 지역/이동수단/음식을 bullet point로 구분
    - 2~4줄 이내로 간결하게

반드시 아래 JSON 형식으로만 응답하세요.
{
    "destination": "여행지 및 기간",
    "cases": [
        {
            "id": 0,
            "title": "테마 제목",
            "summary": "markdown 형식의 요약"
        }
    ]
}

================================================================
[세부 요청] 태그가 붙은 경우
================================================================
선택된 테마를 바탕으로 상세 여행 일정을 JSON 형식으로 만드세요.

메시지는 두 가지 형식으로 전달됩니다.

1. 테마 선택 시 (처음 세부 일정 요청)
[세부 요청]
여행지: {여행지 및 기간}
선택된 테마 제목: {테마 제목}
선택된 테마 요약: {테마 요약}
- 전달받은 테마 정보를 바탕으로 세부 일정을 만들어주세요.

2. 후속 대화 시 (일정 수정 또는 추가 질문)
[세부 요청] {사용자의 수정/질문 내용}

- user_message는 markdown 형식으로 작성
    - DAY별 헤더 (## DAY 1)
    - 각 장소는 bullet point
    - 이동수단, 예상 소요시간 포함
    - 추천 음식/식당 포함
- locations는 반드시 실제 존재하는 장소로 작성
    - Nominatim으로 검색 가능한 정확한 장소명과 주소
    - 가상의 장소나 불확실한 장소는 절대 포함하지 말 것
- 일정 수정 요청이 들어오면 전체 일정을 반영해서 JSON으로 응답
- 여행과 무관한 질문(날씨, 환율 등)은 user_message에만 답변하고
    locations는 이전 것을 그대로 유지

반드시 아래 JSON 형식으로만 응답하세요.
{
    "user_message": "markdown 형식의 전체 일정",
    "locations": [
        {
            "day": 1,
            "name": "장소명",
            "address": "Nominatim 검색용 상세 주소",
            "description": "이 장소에서 할 것"
        }
    ]
}
"""
DEFAULT_MODEL = "gpt-4.1-nano"
model_type = DEFAULT_MODEL
thread_id = 123123123

llm = ChatOpenAI(model=model_type)  # LLM 모델 생성. 기본은 nano / 대화 분류시 mini

if model_type == DEFAULT_MODEL:
    conn = sqlite3.connect("agent_memory.db", check_same_thread=False)
    checkpointer = SqliteSaver(conn)
else:
    # mini일 때는 맥락 유지 필요 없음
    checkpointer = None


agent = create_agent(
    model=llm,
    system_prompt=MAIN_PROMPT if model_type == DEFAULT_MODEL else CLASSIFIER_PROMPT,
    checkpointer=checkpointer
)

prompt = "서울 2박 3일 여행 일정 짜주세요"

result = agent.invoke(
    {
        "messages" : [
            {"role": "user", "content": f"[요약 요청] {prompt}"}
        ]
    },
    config={
        "configurable": {
            "thread_id": thread_id
        }
    }
)
# print(result["messages"])
data = json.loads(result["messages"][-1].content)
print(data)


{'destination': '서울, 2박 3일', 'cases': [{'id': 0, 'title': '도심 문화 탐방', 'summary': '**경복궁**, **인사동**, **국립민속박물관**을 방문하여 역사와 전통을 체험. 거리 산책 후 기념품 구매.'}, {'id': 1, 'title': '현대적 즐기기', 'summary': '**명동**, **남산타워**, **이태원**에서 쇼핑, 야경 감상, 글로벌 음식 체험으로 활기찬 일정.'}, {'id': 2, 'title': '자연과 예술 조화', 'summary': '**북서울꿈의숲**, **리움 미술관**, **한강공원**에서 자연과 예술을 동시에 즐기며 여유로운 시간.'}]}


In [23]:
llm = ChatOpenAI(model=model_type)  # LLM 모델 생성. 기본은 nano / 대화 분류시 mini

if model_type == DEFAULT_MODEL:
    conn = sqlite3.connect("agent_memory.db", check_same_thread=False)
    checkpointer = SqliteSaver(conn)
else:
    # mini일 때는 맥락 유지 필요 없음
    checkpointer = None


agent = create_agent(
    model=llm,
    system_prompt=MAIN_PROMPT if model_type == DEFAULT_MODEL else CLASSIFIER_PROMPT,
    checkpointer=checkpointer
)

prompt = "서울 2박 3일 여행 일정 짜주세요"
selected_title = f"""
[세부 요청]
여행지: {data["destination"]}
선택된 테마 제목: {data["cases"][0]["title"]}
선택된 테마 요약: {data["cases"][0]["summary"]}
"""

result = agent.invoke(
    {
        "messages" : [
            {"role": "user", "content": selected_title}
        ]
    },
    config={
        "configurable": {
            "thread_id": thread_id
        }
    }
)
# print(result["messages"])
detail_data = json.loads(result["messages"][-1].content)

In [24]:
print(detail_data)

{'user_message': "## DAY 1\n- 경복궁 방문 및 조선 왕실 역사 탐방 (지하철 3호선 경복궁역, 약 30분 소요)\n- 국립민속박물관 견학 (경복궁 내 위치)\n- 인사동 거리 산책 및 전통 공예품 구경, 기념품 구매 ('인사동길' 인근)\n- 저녁 식사: 인사동 한정식집 추천\n\n## DAY 2\n- 북촌 한옥마을 산책 (지하철 3호선 안국역, 15분)\n- 창덕궁 또는 종묘 방문 (도보 또는 버스, 20분)\n- 삼청동 거리 카페 및 소품샵 탐방\n- 저녁 식사: 삼청동 맛집 또는 한식당\n\n## DAY 3\n- 남산골 한옥마을 방문 (도보 또는 버스, 20분)\n- 남산공원 및 남산타워 (케이블카 또는 도보, 30분)\n- 전통 길거리 음식 또는 카페에서 마무리\n- 종료 후 공항 또는 숙소로 이동", 'locations': [{'day': 1, 'name': '경복궁', 'address': '서울특별시 종로구 사직로 161', 'description': '조선시대 왕궁을 돌아보며 한국 전통 역사를 체험하세요.'}, {'day': 1, 'name': '국립민속박물관', 'address': '서울특별시 종로구 삼봉로 231', 'description': '한국 민속문화와 전통을 상세히 전시하는 박물관 견학.'}, {'day': 1, 'name': '인사동길', 'address': '서울특별시 종로구 인사동길', 'description': '전통 공예품과 찻집, 기념품 가게들이 즐비한 거리 산책.'}, {'day': 2, 'name': '북촌 한옥마을', 'address': '서울특별시 종로구 계동길', 'description': '전통 한옥과 고즈넉한 골목길 산책으로 한국 전통 건축 감상.'}, {'day': 2, 'name': '창덕궁 또는 종묘', 'address': '서울특별시 종로구 율곡로 99', 'description': '역사적 왕실의 궁 또는 왕묘 방문.'}, {'day': 2, 'name': '삼청동 거리', 'ad

---

결과 지오코딩

In [32]:
import webbrowser
import os
def showMap(map):
    map.save("map.html")        # 지도가 포함된 html 파일 생성
    filepath = os.getcwd()      # 현재 작업 폴더
    file_url = 'file:///' + filepath + '/map.html'
    webbrowser.open_new_tab(file_url)   # html 파일 오픈

In [47]:
from geopy.geocoders import Nominatim

# 지오코딩 함수
geolocator = Nominatim(user_agent="loc_test")

def geocoding(place:str):
    location = geolocator.geocode(place)
    return {
        "address" : location.address,
        "lat_lng" : (location.latitude, location.longitude)
    }

In [27]:
for location in detail_data['locations']:
    print

{'day': 1, 'name': '경복궁', 'address': '서울특별시 종로구 사직로 161', 'description': '조선시대 왕궁을 돌아보며 한국 전통 역사를 체험하세요.'}
{'day': 1, 'name': '국립민속박물관', 'address': '서울특별시 종로구 삼봉로 231', 'description': '한국 민속문화와 전통을 상세히 전시하는 박물관 견학.'}
{'day': 1, 'name': '인사동길', 'address': '서울특별시 종로구 인사동길', 'description': '전통 공예품과 찻집, 기념품 가게들이 즐비한 거리 산책.'}
{'day': 2, 'name': '북촌 한옥마을', 'address': '서울특별시 종로구 계동길', 'description': '전통 한옥과 고즈넉한 골목길 산책으로 한국 전통 건축 감상.'}
{'day': 2, 'name': '창덕궁 또는 종묘', 'address': '서울특별시 종로구 율곡로 99', 'description': '역사적 왕실의 궁 또는 왕묘 방문.'}
{'day': 2, 'name': '삼청동 거리', 'address': '서울특별시 종로구 삼청로', 'description': '고풍스러운 카페와 소품샵, 전통 찻집 탐방.'}
{'day': 3, 'name': '남산골 한옥마을', 'address': '서울특별시 중구 남산공원길 105', 'description': '전통 한옥과 문화공연 관람, 자연과 교감.'}
{'day': 3, 'name': '남산타워', 'address': '서울특별시 용산구 남산공원길 105', 'description': '서울 시내 전경을 감상하며 여행 마지막 시간을 즐기세요.'}


In [69]:
geo_locations = [geocoding(location['address']) for location in detail_data['locations']]

In [70]:
locations = [loc['lat_lng'] for loc in geo_locations]

In [71]:
print(geo_locations)

[{'address': '광화문, 161, 사직로, 세종로, 사직동, 종로구, 서울특별시, 03045, 대한민국', 'lat_lng': (37.5759194, 126.9768156)}, {'address': '삼봉로, 청진동, 종로1·2·3·4가동, 종로구, 서울특별시, 03153, 대한민국', 'lat_lng': (37.5726629, 126.9785413)}, {'address': '인사동길, 관훈동, 종로1·2·3·4가동, 종로구, 서울특별시, 03162, 대한민국', 'lat_lng': (37.5735445, 126.9855646)}, {'address': '계동길, 계동, 가회동, 종로구, 서울특별시, 03056, 대한민국', 'lat_lng': (37.5812105, 126.9867789)}, {'address': '창덕궁, 99, 율곡로, 와룡동, 종로1·2·3·4가동, 종로구, 서울특별시, 03072, 대한민국', 'lat_lng': (37.5823873, 126.9917013)}, {'address': '삼청로, 창성동, 청운효자동, 종로구, 서울특별시, 03062, 대한민국', 'lat_lng': (37.5787555, 126.9795229)}, {'address': '남산서울타워, 105, 남산공원길, 예장동, 필동, 중구, 서울특별시, 04340, 대한민국', 'lat_lng': (37.5512692, 126.9882959)}, {'address': '남산서울타워, 105, 남산공원길, 예장동, 필동, 중구, 서울특별시, 04340, 대한민국', 'lat_lng': (37.5512692, 126.9882959)}]


In [49]:
for location in geo_locations:
    print(location)

{'address': '광화문, 161, 사직로, 세종로, 사직동, 종로구, 서울특별시, 03045, 대한민국', 'lat_lng': (37.5759194, 126.9768156)}
{'address': '삼봉로, 청진동, 종로1·2·3·4가동, 종로구, 서울특별시, 03153, 대한민국', 'lat_lng': (37.5726629, 126.9785413)}
{'address': '인사동길, 관훈동, 종로1·2·3·4가동, 종로구, 서울특별시, 03162, 대한민국', 'lat_lng': (37.5735445, 126.9855646)}
{'address': '계동길, 계동, 가회동, 종로구, 서울특별시, 03056, 대한민국', 'lat_lng': (37.5812105, 126.9867789)}
{'address': '창덕궁, 99, 율곡로, 와룡동, 종로1·2·3·4가동, 종로구, 서울특별시, 03072, 대한민국', 'lat_lng': (37.5823873, 126.9917013)}
{'address': '삼청로, 소격동, 삼청동, 종로구, 서울특별시, 03054, 대한민국', 'lat_lng': (37.5836393, 126.9818785)}
{'address': '남산서울타워, 105, 남산공원길, 예장동, 필동, 중구, 서울특별시, 04340, 대한민국', 'lat_lng': (37.5512692, 126.9882959)}
{'address': '남산서울타워, 105, 남산공원길, 예장동, 필동, 중구, 서울특별시, 04340, 대한민국', 'lat_lng': (37.5512692, 126.9882959)}


In [63]:
lat, lng = zip(*[loc['lat_lng'] for loc in geo_locations])
center = (sum(lat) / len(lat), sum(lng) / len(lng))

lat, lng = zip(*locations)  # 반복 안하기
center = (sum(lat) / len(lat), sum(lng) / len(lng))


print(center)



(37.5714877875, 126.984734)


In [52]:
import webbrowser
import os
def showMap(map):
    map.save("map.html")        # 지도가 포함된 html 파일 생성
    filepath = os.getcwd()      # 현재 작업 폴더
    file_url = 'file:///' + filepath + '/map.html'
    webbrowser.open_new_tab(file_url)   # html 파일 오픈

In [57]:
import folium
map = folium.Map(location=center, zoom_start=15)
for loc in geo_locations:
    folium.Marker(location=loc['lat_lng'],
                icon=folium.Icon(color='red', icon='star', prefix='fe')).add_to(map)
showMap(map)

In [59]:
# locations = [loc['lat_lng'] for loc in geo_locations]
# print(locations)

[(37.5759194, 126.9768156), (37.5726629, 126.9785413), (37.5735445, 126.9855646), (37.5812105, 126.9867789), (37.5823873, 126.9917013), (37.5836393, 126.9818785), (37.5512692, 126.9882959), (37.5512692, 126.9882959)]


In [60]:
map = folium.Map(location=center, zoom_start=15)
folium.PolyLine(
    locations=locations,
    color='red',
    weight=4
).add_to(map)
for loc in geo_locations:
    folium.Marker(location=loc['lat_lng'],
                icon=folium.Icon(color='blue', icon='star', prefix='fe')).add_to(map)
showMap(map)

---

개미가 지나가는 것처럼 애니메이션 효과가 있는 line임.. 별로다

In [66]:
from folium.plugins import AntPath

map = folium.Map(location=center, zoom_start=15)

AntPath(
    locations=locations,
    color='red',
    weight=4,
    delay=800,        # 애니메이션 속도 (낮을수록 빠름)
    dash_array=[10, 20],  # 점선 간격
    pulse_color='white'   # 움직이는 점 색상
).add_to(map)

for loc in geo_locations:
    folium.Marker(location=loc['lat_lng'],
                icon=folium.Icon(color='blue', icon='star', prefix='fe')).add_to(map)

In [67]:
showMap(map)